In [1]:
import csv
import json
import os
import sys
import types
from datetime import datetime, timedelta, timezone
from importlib.metadata import version
from pathlib import Path

from dotenv import load_dotenv

import cattr
import httpx._types as httpx_types

if not hasattr(httpx_types, "VerifyTypes"):
    httpx_types.VerifyTypes = bool | str | Path

if "pkg_resources" not in sys.modules:
    pkg_resources = types.ModuleType("pkg_resources")

    class _Distribution:
        def __init__(self, package_name: str):
            self.version = version(package_name)

    pkg_resources.get_distribution = _Distribution
    sys.modules["pkg_resources"] = pkg_resources

if not hasattr(cattr.converters.BaseConverter, "_unstructure_enum"):
    def _unstructure_enum(self, obj):
        return obj.value

    cattr.converters.BaseConverter._unstructure_enum = _unstructure_enum

import toloka.client as toloka
import toloka.client.project.template_builder as tb


In [2]:
load_dotenv()

TOKEN = os.getenv("TOLOKA_TOKEN")
TOLOKA_API_URL = "https://tasks.yandex.ru"
BBOX_PROJECT_NAME = "[bbox] Сегментация бабочек"
BBOX_PROJECT_ID = "9642"
TARGET_BBOX_RESPONSES = 200

bbox_interface_path = "data/bbox-interface.json"
bbox_instruction_path = "data/bbox-instruction.html"
bbox_exam_tsv_path = "data/bbox_project_exam.tsv"

toloka_client = toloka.TolokaClient(TOKEN, url=TOLOKA_API_URL)


## Проверка доступа

In [3]:
requester = toloka_client.get_requester()
print("public name:", getattr(requester, "public_name", None))
print("balance:", getattr(requester, "balance", None))


public name: {'EN': 'eraserhead@aiconf-2026'}
balance: 1491.4000


## Создание bbox-проекта

In [4]:
with open(bbox_interface_path, encoding="utf-8") as file:
    bbox_interface_config = json.load(file)

with open(bbox_instruction_path, encoding="utf-8") as file:
    bbox_project_instructions = file.read()


bbox_view_spec = toloka.project.TemplateBuilderViewSpec(
    config=tb.TemplateBuilder.structure(bbox_interface_config),
)

if BBOX_PROJECT_ID:
    bbox_project = toloka_client.get_project(BBOX_PROJECT_ID)
else:
    bbox_project = toloka.project.Project(
        public_name=BBOX_PROJECT_NAME,
        public_description="Проект для разметки бабочек на изображениях.",
        public_instructions=bbox_project_instructions,
        task_spec=toloka.project.task_spec.TaskSpec(
            input_spec={
                "image": toloka.project.field_spec.UrlSpec(required=True),
                "taxon": toloka.project.field_spec.StringSpec(required=False),
                "photo_id": toloka.project.field_spec.StringSpec(required=False),
                "hard": toloka.project.field_spec.StringSpec(required=False),
            },
            output_spec={
                "result": toloka.project.field_spec.JsonSpec(required=False),
                "no_object": toloka.project.field_spec.BooleanSpec(required=False),
            },
            view_spec=bbox_view_spec,
        ),
    )

    bbox_project = toloka_client.create_project(bbox_project)
bbox_project


Project(_unexpected={'public_instructions_price': Decimal('0.00'), 'service_type': 'UNDEFINED', 'project_types': [], 'content_types': [], 'owner': {'id': '4f61e1bd6deac48f1628b09918db2205', 'myself': True, 'company_id': '151'}}, public_name='[bbox] Сегментация бабочек', public_description='Проект для разметки бабочек на изображениях.', task_spec=TaskSpec(_unexpected={}, input_spec={'hard': StringSpec(_unexpected={}, required=False, hidden=False, min_length=None, max_length=None, allowed_values=None), 'image': UrlSpec(_unexpected={}, required=True, hidden=False), 'taxon': StringSpec(_unexpected={}, required=False, hidden=False, min_length=None, max_length=None, allowed_values=None), 'photo_id': StringSpec(_unexpected={}, required=False, hidden=False, min_length=None, max_length=None, allowed_values=None)}, output_spec={'result': JsonSpec(_unexpected={}, required=False, hidden=False), 'no_object': BooleanSpec(_unexpected={}, required=False, hidden=False, allowed_values=None)}, view_spec=

## Создание экзаменационного bbox-пула

Это обычный пул с постприёмкой: исполнители рисуют bbox, а дальше мы отдельно отправим их ответы в review-проект на проверку.

In [5]:
with open(bbox_exam_tsv_path, encoding="utf-8", newline="") as file:
    bbox_exam_rows = list(csv.DictReader(file, delimiter="	"))

bbox_exam_tasks_count = len(bbox_exam_rows)

bbox_exam_pool = toloka.pool.Pool(
    project_id=bbox_project.id,
    private_name=f"bbox_exam_{datetime.now().strftime('%Y%m%d_%H%M%S')}",
    may_contain_adult_content=False,
    reward_per_assignment=0.25,
    assignment_max_duration_seconds=20 * 60,
    will_expire=datetime.now(timezone.utc) + timedelta(days=30),
    auto_accept_solutions=False,
    defaults=toloka.pool.Pool.Defaults(
        default_overlap_for_new_task_suites=TARGET_BBOX_RESPONSES,
    ),
    filter=(
        (toloka.filter.ClientType == "BROWSER")
    ) & (
        toloka.filter.Languages.in_("RU")
    ) & toloka.filter.FilterOr([
        toloka.filter.RegionByPhone.in_(225),
        toloka.filter.RegionByIp.in_(225),
    ]),
)

bbox_exam_pool.set_mixer_config(
    real_tasks_count=bbox_exam_tasks_count,
    golden_tasks_count=0,
    training_tasks_count=0,
    min_real_tasks_count=bbox_exam_tasks_count,
    force_last_assignment=True,
    mix_tasks_in_creation_order=False,
    shuffle_tasks_in_task_suite=False,
)

bbox_exam_pool = toloka_client.create_pool(bbox_exam_pool)
print("project_id:", bbox_project.id)
print("pool_id:", bbox_exam_pool.id)


RetryError: HTTPSConnectionPool(host='tasks.yandex.ru', port=443): Max retries exceeded with url: /api/v1/pools (Caused by ResponseError('too many 500 error responses'))

## Загрузка заданий из bbox_project_exam.tsv

In [ ]:
bbox_exam_tasks = []

for row in bbox_exam_rows:
    bbox_exam_tasks.append(
        toloka.Task(
            pool_id=bbox_exam_pool.id,
            input_values={
                "image": row["INPUT:image"],
                "taxon": row["INPUT:taxon"],
                "photo_id": row["INPUT:photo_id"],
                "hard": row["INPUT:hard"],
            },
        )
    )

bbox_exam_upload_result = toloka_client.create_tasks(bbox_exam_tasks, allow_defaults=True)
print("Заданий загружено:", len(bbox_exam_upload_result.items))


100%|██████████| 100/100 [00:02<00:00, 47.48it/s]


Заданий загружено: 10


## Идем в ЯЗ, жмем "Отправить на модерацию"...

## Проверка статуса проекта и пула

In [ ]:
fresh_project = toloka_client.get_project(bbox_project.id)
fresh_pool = toloka_client.get_pool(bbox_exam_pool.id)

print("project_status:", getattr(fresh_project, "status", None))
print("pool_status:", getattr(fresh_pool, "status", None))
print("last_close_reason:", getattr(fresh_pool, "last_close_reason", None))


project_status: ProjectStatus.ACTIVE
pool_status: Status.CLOSED
last_close_reason: CloseReason.COMPLETED


## Подготовка review-пула внутри существующего review-проекта

После того как bbox-пул собрал ответы, здесь можно указать `review_project_id`, создать в нём review-пул и перелить туда bbox-результаты.

In [ ]:
review_project_id = "" #!!!
review_project_skill_id = "" #!!!
review_pool_overlap = 3
review_pool_reward = 0.5
review_pool_assignment_max_duration_seconds = 10 * 60

bbox_exam_pool_id_for_review_upload = "" #!!!

In [ ]:
if bbox_exam_pool_id_for_review_upload:
    bbox_exam_pool = toloka_client.get_pool(bbox_exam_pool_id_for_review_upload)
    print("using existing bbox_exam_pool:", bbox_exam_pool.id)
else:
    print("using bbox_exam_pool from notebook state:", bbox_exam_pool.id)


using existing bbox_exam_pool: 6123522


In [ ]:
review_pool = toloka.pool.Pool(
    project_id=review_project_id,
    private_name=f"bbox_review_{datetime.now().strftime('%Y%m%d_%H%M%S')}",
    may_contain_adult_content=False,
    reward_per_assignment=review_pool_reward,
    assignment_max_duration_seconds=review_pool_assignment_max_duration_seconds,
    will_expire=datetime.now(timezone.utc) + timedelta(days=30),
    auto_accept_solutions=True,
    defaults=toloka.pool.Pool.Defaults(
        default_overlap_for_new_task_suites=review_pool_overlap,
    ),
    mixer_config=toloka.pool.Pool.MixerConfig(
        real_tasks_count=10,
        golden_tasks_count=0,
        training_tasks_count=0,
        min_real_tasks_count=10,
        force_last_assignment=True,
        mix_tasks_in_creation_order=False,
        shuffle_tasks_in_task_suite=False,
    ),
    quality_control=toloka.quality_control.QualityControl(
        captcha_frequency=toloka.quality_control.QualityControl.CaptchaFrequency.LOW,
    ),
    filter=(
        (toloka.filter.ClientType == "BROWSER")
    ) & (
        toloka.filter.Languages.in_("RU")
    ) & (
        toloka.filter.Skill(review_project_skill_id) >= 100
    ) & toloka.filter.FilterOr([
        toloka.filter.RegionByPhone.in_(225),
        toloka.filter.RegionByIp.in_(225),
    ]),
)

review_pool.quality_control.add_action(
    collector=toloka.collectors.Captcha(history_size=8),
    conditions=[
        toloka.conditions.StoredResultsCount >= 8,
        toloka.conditions.FailRate >= 50,
    ],
    action=toloka.actions.RestrictionV2(
        scope="PROJECT",
        duration=365,
        duration_unit="DAYS",
        private_comment="4 из 8 капч решены неверно",
    ),
)

review_pool = toloka_client.create_pool(review_pool)
print("review_pool_id:", review_pool.id)


CAPTCHA-based quality control settings are deprecated and will be removed in future. CAPTCHAs are now included automatically for better quality control.
CAPTCHA-based quality control settings are deprecated and will be removed in future. CAPTCHAs are now included automatically for better quality control.


review_pool_id: 6124122


Собираем все выполненные bbox assignments и для каждой пары `task/solution` создаем отдельную задачу в review-пуле.
Это важно: один bbox assignment содержит весь набор экзаменационных картинок, поэтому здесь нужно проходить по всем задачам внутри assignment, а не только по первой.

In [ ]:
bbox_assignments = list(toloka_client.get_assignments(pool_id=bbox_exam_pool.id))
print("bbox assignments:", len(bbox_assignments))

review_tasks = []
review_task_mapping = []

for assignment in bbox_assignments:
    tasks = assignment.tasks or []
    solutions = assignment.solutions or []

    if not tasks or not solutions:
        continue

    for task_index, (task, solution) in enumerate(zip(tasks, solutions)):
        input_values = task.input_values or {}
        output_values = solution.output_values or {}

        guides = output_values.get("result") or []
        if isinstance(guides, str):
            guides = json.loads(guides)
        if isinstance(guides, dict):
            guides = [guides]

        review_input_values = {
            "image": input_values["image"],
            "guides": guides,
        }

        for field_name in ("hard", "taxon", "photo_id"):
            field_value = input_values.get(field_name)
            if field_value not in (None, ""):
                review_input_values[field_name] = str(field_value)

        review_tasks.append(
            toloka.Task(
                pool_id=review_pool.id,
                input_values=review_input_values,
                unavailable_for=[assignment.user_id] if assignment.user_id else None,
            )
        )

        review_task_mapping.append(
            {
                "bbox_assignment_id": assignment.id,
                "bbox_worker_id": assignment.user_id,
                "bbox_pool_id": bbox_exam_pool.id,
                "bbox_task_index": task_index,
                "photo_id": review_input_values.get("photo_id"),
                "image": review_input_values["image"],
            }
        )

print("review tasks prepared:", len(review_tasks))
print("unique bbox assignments:", len({row["bbox_assignment_id"] for row in review_task_mapping}))
print("unique images:", len({row["image"] for row in review_task_mapping}))
print(review_tasks[0].input_values)


bbox assignments: 150
review tasks prepared: 1000
unique bbox assignments: 100
unique images: 10
{'image': 'https://inaturalist-open-data.s3.amazonaws.com/photos/619698470/original.jpg', 'guides': [{'shape': 'rectangle', 'left': Decimal('0.45423340961098396'), 'top': Decimal('0.32281337401474697'), 'width': Decimal('0.17048054919908467'), 'height': Decimal('0.2634121535723367')}], 'hard': 'False', 'taxon': 'butterfly', 'photo_id': '619698470'}


## Загрузка review tasks и сохранение маппинга

Этот маппинг потом нужен, чтобы по majority vote понять, какие bbox assignments принимать.

In [ ]:
review_upload_result = toloka_client.create_tasks(review_tasks, allow_defaults=True)

print("uploaded items:", len(review_upload_result.items or {}))
print("validation errors:", len(review_upload_result.validation_errors or {}))

if review_upload_result.validation_errors:
    print(json.dumps(review_upload_result.validation_errors, ensure_ascii=False, indent=2))


100%|██████████| 100/100 [01:37<00:00,  1.03it/s]


uploaded items: 1000
validation errors: 0


In [ ]:
from pathlib import Path

uploaded_tasks = list((review_upload_result.items or {}).values())

for uploaded_task, mapping in zip(uploaded_tasks, review_task_mapping):
    mapping["review_task_id"] = uploaded_task.id
    mapping["review_pool_id"] = review_pool.id

mapping_path = Path(f"bbox_to_review_mapping_{review_pool.id}.json")
with mapping_path.open("w", encoding="utf-8") as file:
    json.dump(review_task_mapping, file, ensure_ascii=False, indent=2)

print("review tasks uploaded:", len(uploaded_tasks))
print("unique uploaded images:", len({row["image"] for row in review_task_mapping}))
print("mapping saved:", mapping_path)

review tasks uploaded: 1000
unique uploaded images: 10
mapping saved: bbox_to_review_mapping_6124122.json
